OSU AI 535, Winter 2026
Group: Jacob Beitler, Jonathan Hotchkiss, Seth Mackovjak

Multicategory model.

Overview:
1) Data preprocessing: Import, structure, split, and augment the data.
2) Model definition: ResNetRS50 binary categorization.
3) Training loop definition: Incorporate appropriate Weights and Bias.
3) Model saving.

In [ ]:
from dataset_builder import bcicd_dataset, category_balance_plot

train_dataset = bcicd_dataset(train=True)
test_dataset = bcicd_dataset(train=False)

category_balance_plot(train_dataset=train_dataset,
                      test_dataset=test_dataset)

In [ ]:
# Jacob Beitler
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms, models

# 5 class mapping from bcicd prefix_dict
prefix_dict = {
    'BA': {'name': 'basophil', 'id': 0},
    'ERB': {'name': 'erythroblast', 'id': 1},
    'MO': {'name': 'monocyte', 'id': 2},
    'MYO': {'name': 'myeloblast', 'id': 3},
    'NGS': {'name': 'seg_neutrophil', 'id': 4},
}

id_to_name = {v['id']: v['name'] for v in prefix_dict.values()}
num_classes = len(id_to_name)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# data loaders (dataset images are imported as tensors already)
train_dataset.transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ConvertImageDtype(torch.float),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
test_dataset.transform = train_dataset.transform

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

# model: ResNet50 adapted for 5 classes
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# training loop
for epoch in range(1, 6):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_acc = correct / total
    print(f'Epoch {epoch} - loss: {epoch_loss:.4f}, test_acc: {test_acc:.4f}')

# Example class name output
print('id_to_name:', id_to_name)